In [4]:
# ============================================================
# TVP-VAR Input Preprocessing Script
# - SOLVPN, Copper, Gold, Silver, DXY, UST10Y, VIX
# - 추가: OIL(DCOILWTICO), GAS(DHHNGSP)
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
BASE_DIR = Path("./original_data")
OUT_DIR = Path("./tvpvar_preprocessed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_MAP = {
    "SOLVPN": "SOLVPN_index.csv",
    "COPPER": "copper_futures.csv",
    "GOLD": "gold_futures.csv",
    "SILVER": "silver_futures.csv",
    "DXY": "dollar_index.csv",
    "UST10Y": "us_10y_bond_yield.csv",
    "VIX": "cboe_vix_index.csv",

    # Energy variables
    "OIL": "DCOILWTICO.csv",
    "GAS": "DHHNGSP.csv",
}

LOG_RETURN_VARS = [
    "SOLVPN",
    "COPPER",
    "GOLD",
    "SILVER",
    "DXY",
    "OIL",
    "GAS",
]

DIFF_VARS = [
    "UST10Y",
    "VIX",
]

START_DATE = "2020-10-10"
END_DATE = "2026-01-12"

FINAL_COLUMNS = [
    "Date",
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "dlog_OIL",
    "dlog_GAS",
    "d_UST10Y",
    "d_VIX",
]

# =========================
# Helper
# =========================
def standardize_colnames(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def find_date_column(df):
    candidates = [
        "Date", "date", "DATE",
        "observation_date", "Observation_Date", "OBSERVATION_DATE",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(f"Date column not found. Available columns: {list(df.columns)}")


def find_value_column(df):
    priority = [
        "Close", "close", "CLOSE",
        "Price", "price", "PRICE",
        "Adj Close", "AdjClose", "adj_close",
        "Last", "last", "LAST",
        "Value", "value", "VALUE",

        # FRED columns
        "DCOILWTICO",
        "DHHNGSP",
    ]

    for c in priority:
        if c in df.columns:
            return c

    date_cols = [
        "Date", "date", "DATE",
        "observation_date", "Observation_Date", "OBSERVATION_DATE",
    ]

    non_date_cols = [c for c in df.columns if c not in date_cols]

    if len(non_date_cols) == 1:
        return non_date_cols[0]

    raise ValueError(f"Value column not found. Available columns: {list(df.columns)}")


def clean_numeric_series(s):
    s = (
        s.astype(str)
         .str.strip()
         .str.replace(",", "", regex=False)
         .str.replace("%", "", regex=False)
    )

    # FRED 결측값 "." 처리
    s = s.replace({".": np.nan, "": np.nan, "nan": np.nan, "NaN": np.nan})

    return pd.to_numeric(s, errors="coerce")


def load_single_series(path, var_name):
    df = pd.read_csv(path)
    df = standardize_colnames(df)

    date_col = find_date_column(df)
    value_col = find_value_column(df)

    out = df[[date_col, value_col]].copy()
    out.columns = ["Date", var_name]

    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out[var_name] = clean_numeric_series(out[var_name])

    out = out.dropna(subset=["Date", var_name])
    out = out.sort_values("Date")
    out = out.drop_duplicates(subset=["Date"])
    out = out.reset_index(drop=True)

    return out


def apply_transform(df, var_name):
    out = df.copy()

    if var_name in LOG_RETURN_VARS:
        out.loc[out[var_name] <= 0, var_name] = np.nan
        out[f"dlog_{var_name}"] = 100.0 * np.log(out[var_name]).diff()

    elif var_name in DIFF_VARS:
        out[f"d_{var_name}"] = out[var_name].diff()

    else:
        raise ValueError(f"Unknown transform rule for variable: {var_name}")

    return out


def apply_date_filter(df, start_date, end_date):
    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)

    out = df[(df["Date"] >= start_dt) & (df["Date"] <= end_dt)].copy()
    out = out.sort_values("Date").reset_index(drop=True)

    return out


def merge_raw_level_series():
    merged = None

    for var_name, file_name in FILE_MAP.items():
        path = BASE_DIR / file_name

        if not path.exists():
            raise FileNotFoundError(f"File not found: {path.resolve()}")

        df = load_single_series(path, var_name)

        if merged is None:
            merged = df
        else:
            merged = pd.merge(merged, df, on="Date", how="outer")

    merged = merged.sort_values("Date").reset_index(drop=True)
    return merged


def build_transformed_input():
    merged = None

    for var_name, file_name in FILE_MAP.items():
        path = BASE_DIR / file_name

        if not path.exists():
            raise FileNotFoundError(f"File not found: {path.resolve()}")

        df = load_single_series(path, var_name)
        df = apply_transform(df, var_name)

        if var_name in LOG_RETURN_VARS:
            keep_cols = ["Date", f"dlog_{var_name}"]
        elif var_name in DIFF_VARS:
            keep_cols = ["Date", f"d_{var_name}"]

        df = df[keep_cols].copy()

        if merged is None:
            merged = df
        else:
            merged = pd.merge(merged, df, on="Date", how="inner")

    merged = apply_date_filter(merged, START_DATE, END_DATE)

    final_df = merged.dropna().copy()
    final_df = final_df.sort_values("Date").reset_index(drop=True)

    missing_cols = [c for c in FINAL_COLUMNS if c not in final_df.columns]
    if missing_cols:
        raise ValueError(f"Missing final columns: {missing_cols}")

    final_df = final_df[FINAL_COLUMNS]

    if final_df.empty:
        raise ValueError("No data left after preprocessing.")

    return final_df


def zscore_standardize(df, exclude_cols=None):
    if exclude_cols is None:
        exclude_cols = ["Date"]

    out = df.copy()
    numeric_cols = [c for c in out.columns if c not in exclude_cols]

    means = out[numeric_cols].mean()
    stds = out[numeric_cols].std(ddof=0)

    zero_std_cols = stds[stds == 0].index.tolist()
    if zero_std_cols:
        raise ValueError(f"Zero-std columns found: {zero_std_cols}")

    out[numeric_cols] = (out[numeric_cols] - means) / stds

    return out


# =========================
# Main
# =========================
def main():
    print("=== TVP-VAR preprocessing started ===")

    # 1) Raw level merged
    raw_level_df = merge_raw_level_series()
    raw_level_df = apply_date_filter(raw_level_df, START_DATE, END_DATE)

    # 하나라도 결측이면 row 전체 제거
    raw_level_before = len(raw_level_df)
    raw_level_df = raw_level_df.dropna(how="any").reset_index(drop=True)
    raw_level_after = len(raw_level_df)

    raw_level_path = OUT_DIR / "tvpvar_raw_level_merged.csv"
    raw_level_df.to_csv(raw_level_path, index=False, encoding="utf-8-sig")

    print("[1] Saved raw level merged")
    print(raw_level_path)
    print("Rows before dropna:", raw_level_before)
    print("Rows after dropna :", raw_level_after)
    print("Dropped rows      :", raw_level_before - raw_level_after)
    print("Columns:", raw_level_df.columns.tolist())

    # 2) Transformed + complete-case + standardized input
    final_df = build_transformed_input()

    # 안전장치: 최종 입력에서도 하나라도 결측이면 제거
    final_before = len(final_df)
    final_df = final_df.dropna(how="any").reset_index(drop=True)
    final_after = len(final_df)

    scaled_df = zscore_standardize(final_df, exclude_cols=["Date"])

    scaled_path = OUT_DIR / "tvpvar_input_scaled.csv"
    scaled_df.to_csv(scaled_path, index=False, encoding="utf-8-sig")

    print("\n[2] Saved TVP-VAR input scaled")
    print(scaled_path)
    print("Rows before dropna:", final_before)
    print("Rows after dropna :", final_after)
    print("Dropped rows      :", final_before - final_after)
    print("Columns:", scaled_df.columns.tolist())

    print("\nDate range:", scaled_df["Date"].min(), "~", scaled_df["Date"].max())
    print(f"Saved to: {OUT_DIR.resolve()}")
    print("=== TVP-VAR preprocessing finished ===")


if __name__ == "__main__":
    main()

=== TVP-VAR preprocessing started ===
[1] Saved raw level merged
tvpvar_preprocessed\tvpvar_raw_level_merged.csv
Rows before dropna: 1401
Rows after dropna : 1033
Dropped rows      : 368
Columns: ['Date', 'SOLVPN', 'COPPER', 'GOLD', 'SILVER', 'DXY', 'UST10Y', 'VIX', 'OIL', 'GAS']

[2] Saved TVP-VAR input scaled
tvpvar_preprocessed\tvpvar_input_scaled.csv
Rows before dropna: 1032
Rows after dropna : 1032
Dropped rows      : 0
Columns: ['Date', 'dlog_SOLVPN', 'dlog_COPPER', 'dlog_GOLD', 'dlog_SILVER', 'dlog_DXY', 'dlog_OIL', 'dlog_GAS', 'd_UST10Y', 'd_VIX']

Date range: 2020-10-13 00:00:00 ~ 2026-01-06 00:00:00
Saved to: D:\University\3-2.5\PADA_Lab\DC_and_meterials_2\02_Data_EDA\tvpvar_preprocessed
=== TVP-VAR preprocessing finished ===
